# **Calibração dos sensores**

In [1]:
# Execute esta célula e cole o resultado
PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

print(f"Conteúdo da pasta {PASTA_DRIVE}:")
!ls "$PASTA_DRIVE"

Conteúdo da pasta /content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao:
Arduino.txt			 Casa_2.txt	       Mata.TXT
Calculo_2_Media_Ponderada.ipynb  Casa_estruturado.csv  Mato_2_estruturado.csv
Calculo_Media_Ponderada.ipynb	 Casa.txt	       Mato_2.txt
Casa_2_estruturado.csv		 Mata_estruturado.csv  Raspberry.txt


In [6]:
import pandas as pd
import os
from google.colab import drive
import numpy as np
import re

# ====================================================================
# 1. Montagem e Definição de Caminhos
# ====================================================================

print("Montando Google Drive...")
if not os.path.exists('/content/drive'):
    drive.mount('/content/drive')
else:
    print("Drive já montado.")

PASTA_DRIVE = '/content/drive/MyDrive/Colab Notebooks/Projeto_dissertacao'

# Nomes dos arquivos de entrada no Drive
ARQUIVO_ARDUINO_TXT = 'Arduino.txt'
ARQUIVO_RASPBERRY_TXT = 'Raspberry.txt'

# Caminhos de saída estruturada
CAMINHO_SAVE_ARDUINO = os.path.join(PASTA_DRIVE, 'arduino_estruturado.csv')
CAMINHO_SAVE_RASPBERRY = os.path.join(PASTA_DRIVE, 'raspberry_estruturado.csv')

# ====================================================================
# 2. FUNÇÕES DE ESTRUTURAÇÃO (Inalteradas)
# ====================================================================

def processar_dataframe(df, nome_log, caminho_saida):
    """Função auxiliar para limpeza e salvamento final do DataFrame."""
    df.dropna(subset=['Timestamp', 'Dispositivo'], inplace=True)
    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza inicial.")
        return False

    df.set_index('Timestamp', inplace=True)

    # Forçar tipos numéricos
    df['Temperatura'] = pd.to_numeric(df['Temperatura'], errors='coerce')
    df['Umidade'] = pd.to_numeric(df['Umidade'], errors='coerce')
    df.dropna(subset=['Temperatura', 'Umidade'], inplace=True)

    if df.empty:
        print(f"AVISO: O DataFrame de {nome_log} ficou vazio após a limpeza de Temperatura/Umidade.")
        return False

    # Salvar
    df[['Dispositivo', 'Temperatura', 'Umidade']].to_csv(caminho_saida)
    print(f"Estruturação de {nome_log} concluída. Salvo em: {os.path.basename(caminho_saida)} (Total de {len(df)} linhas)")
    return True

def estruturar_log_regex(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando o padrão REGEX."""
    print(f"\nIniciando estruturação (REGEX) de: {os.path.basename(caminho_entrada)}")
    padrao = re.compile(r"\[(?P<Timestamp>\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2})\].*?Dispositivo: (?P<Dispositivo>[^,]+).*?Temp: (?P<Temp>[\d\.]+).*?Umid: (?P<Umid>[\d\.]+)")

    dados = []
    try:
        with open(caminho_entrada, 'r', encoding='utf-8') as f:
            for linha in f:
                match = padrao.search(linha)
                if match:
                    dados.append(match.groupdict())
    except FileNotFoundError:
        return False
    except Exception as e:
        print(f"ERRO de leitura REGEX: {e}")
        return False

    df = pd.DataFrame(dados)
    if df.empty:
        print(f"AVISO REGEX: Nenhuma linha de dados válida encontrada em {nome_log}.")
        return False

    df['Timestamp'] = pd.to_datetime(df['Timestamp'], format='%Y-%m-%d %H:%M:%S', errors='coerce')
    df.rename(columns={'Temp': 'Temperatura', 'Umid': 'Umidade'}, inplace=True)

    return processar_dataframe(df, nome_log, caminho_saida)

def estruturar_log_tabular(caminho_entrada, caminho_saida, nome_log):
    """Tenta estruturar usando separador TAB (\t)."""
    print(f"\nTentando estruturação (TABULAR) de: {os.path.basename(caminho_entrada)}")
    try:
        df = pd.read_csv(caminho_entrada, sep='\t', header=None,
                         names=['Dispositivo', 'Data', 'Hora', 'Temperatura', 'Umidade'])
    except Exception:
        print(f"AVISO TABULAR: Falha na leitura TABULAR de {nome_log}. O formato do arquivo é diferente.")
        return False

    if 'Data' not in df.columns or 'Hora' not in df.columns:
        return False

    df['Timestamp'] = df['Data'].astype(str) + ' ' + df['Hora'].astype(str)
    df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', infer_datetime_format=True)

    return processar_dataframe(df, nome_log, caminho_saida)


# --- EXECUÇÃO DA ESTRUTURAÇÃO (TENTATIVA MÚLTIPLA) ---

# Tenta estruturar o ARDUINO com REGEX, se falhar, tenta TABULAR
if not estruturar_log_regex(os.path.join(PASTA_DRIVE, ARQUIVO_ARDUINO_TXT), CAMINHO_SAVE_ARDUINO, 'ARDUINO'):
    estruturar_log_tabular(os.path.join(PASTA_DRIVE, ARQUIVO_ARDUINO_TXT), CAMINHO_SAVE_ARDUINO, 'ARDUINO')

# Tenta estruturar o RASPBERRY com REGEX, se falhar, tenta TABULAR
if not estruturar_log_regex(os.path.join(PASTA_DRIVE, ARQUIVO_RASPBERRY_TXT), CAMINHO_SAVE_RASPBERRY, 'RASPBERRY'):
    estruturar_log_tabular(os.path.join(PASTA_DRIVE, ARQUIVO_RASPBERRY_TXT), CAMINHO_SAVE_RASPBERRY, 'RASPBERRY')


# ====================================================================
# 3. Carregamento Final
# ====================================================================

df_arduino = pd.DataFrame()
df_raspberry = pd.DataFrame()

try:
    # Carregar df_raspberry (Baseline/Pico W 1)
    df_raspberry = pd.read_csv(CAMINHO_SAVE_RASPBERRY, index_col=0, parse_dates=True)
    df_raspberry.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_raspberry['Temp'] = pd.to_numeric(df_raspberry['Temp'], errors='coerce')
    df_raspberry['Umid'] = pd.to_numeric(df_raspberry['Umid'], errors='coerce')
    df_raspberry.dropna(subset=['Temp', 'Umid'], inplace=True)

    # Carregar df_arduino (Referência/Arduino)
    df_arduino = pd.read_csv(CAMINHO_SAVE_ARDUINO, index_col=0, parse_dates=True)
    df_arduino.rename(columns={'Temperatura': 'Temp', 'Umidade': 'Umid'}, inplace=True)
    df_arduino['Temp'] = pd.to_numeric(df_arduino['Temp'], errors='coerce')
    df_arduino['Umid'] = pd.to_numeric(df_arduino['Umid'], errors='coerce')
    df_arduino.dropna(subset=['Temp', 'Umid'], inplace=True)

except Exception:
    pass # O erro será tratado pela checagem de DataFrames vazios

# Verifica se os dataframes estão vazios
if df_raspberry.empty or df_arduino.empty:
    print("\nERRO FATAL: Um ou ambos DataFrames estão vazios. Verifique o formato dos arquivos TXT.")
    exit()
else:
    print("\nDataFrames ARDUINO e RASPBERRY carregados com sucesso. Iniciando Análise.")

# ====================================================================
# 4. Recálculo das Médias Ponderadas (Análise)
# ====================================================================

# ----------------------------------------------------
# A) Média Ponderada para ARDUINO (Referência/Baseline)
# ----------------------------------------------------
# Assumimos que o log do Arduino contém apenas um dispositivo,
# mas se contiver mais de um, a média é feita sobre todos.
duracao_arduino_geral = df_arduino.index.to_series().diff().shift(-1).ffill()
pesos_arduino_geral = duracao_arduino_geral.dt.total_seconds()

media_temp_arduino_ref = np.average(df_arduino['Temp'], weights=pesos_arduino_geral)
media_umid_arduino_ref = np.average(df_arduino['Umid'], weights=pesos_arduino_geral)


# ----------------------------------------------------
# B) Médias Ponderadas para RASPBERRY (Múltiplos Dispositivos)
# ----------------------------------------------------
resultados_raspberry = []
grupos_dispositivo_raspberry = df_raspberry.groupby('Dispositivo')

for dispositivo, df_grupo in grupos_dispositivo_raspberry:
    if len(df_grupo) < 1:
        continue

    duracao_raspberry = df_grupo.index.to_series().diff().shift(-1).ffill()
    pesos_raspberry = duracao_raspberry.dt.total_seconds()

    media_temp_raspberry = np.average(df_grupo['Temp'], weights=pesos_raspberry)
    media_umid_raspberry = np.average(df_grupo['Umid'], weights=pesos_raspberry)

    resultados_raspberry.append({
        'Dispositivo': dispositivo,
        'Temp_Média_Ponderada': media_temp_raspberry,
        'Umid_Média_Ponderada': media_umid_raspberry
    })

df_resultados_raspberry = pd.DataFrame(resultados_raspberry).set_index('Dispositivo')


# ====================================================================
# 5. Apresentação e Comparação (Arduino como Baseline)
# ====================================================================

# ----------------------------------------------------
# NOVO: SAÍDA DOS RESULTADOS INDIVIDUAIS
# ----------------------------------------------------
print("\n" + "="*90)
# Média do ARDUINO (A nova Referência)
print(f"|  Média Ponderada do ARDUINO (Referência): Temp: {media_temp_arduino_ref:.2f}°C | Umid: {media_umid_arduino_ref:.2f}%  |")
print("="*90 + "\n")

print("-" * 90)
print("--- 🔬 Médias Ponderadas dos RASPBERRY PICO W ---")
print("-" * 90)

# Exibe as médias individuais dos dispositivos Raspberry
for index, row in df_resultados_raspberry.iterrows():
    print(f"|  Média Ponderada do {index}: Temp: {row['Temp_Média_Ponderada']:.2f}°C | Umid: {row['Umid_Média_Ponderada']:.2f}%  |")
print("-" * 90)


# ----------------------------------------------------
# NOVO: CÁLCULO DAS DIFERENÇAS (Raspberry Pico W - Arduino)
# ----------------------------------------------------
# Cria as colunas de diferença usando o Arduino como referência
df_resultados_raspberry['Diferença_Temp_vs_Arduino (°C)'] = df_resultados_raspberry['Temp_Média_Ponderada'] - media_temp_arduino_ref
df_resultados_raspberry['Diferença_Umid_vs_Arduino (%)'] = df_resultados_raspberry['Umid_Média_Ponderada'] - media_umid_arduino_ref


# Tabela de Comparação
df_comparacao_final = df_resultados_raspberry[[
    'Diferença_Temp_vs_Arduino (°C)',
    'Diferença_Umid_vs_Arduino (%)'
]].round(2)

# Renomeia o índice para ficar mais claro na saída
df_comparacao_final.index.name = 'Dispositivo'
df_comparacao_final.reset_index(inplace=True)
df_comparacao_final['Dispositivo'] = 'Arduino x ' + df_comparacao_final['Dispositivo']
df_comparacao_final.set_index('Dispositivo', inplace=True)


print("\n--- 📊 Comparação de Médias (Dispositivo - Arduino) 📊 ---")
print("   (Valores Positivos: Mais Alto que o Arduino)")
print("   (Valores Negativos: Mais Baixo que o Arduino)")
print("-" * 60)
print(df_comparacao_final)
print("-" * 60)

Montando Google Drive...
Drive já montado.

Iniciando estruturação (REGEX) de: Arduino.txt
AVISO REGEX: Nenhuma linha de dados válida encontrada em ARDUINO.

Tentando estruturação (TABULAR) de: Arduino.txt
Estruturação de ARDUINO concluída. Salvo em: arduino_estruturado.csv (Total de 341 linhas)

Iniciando estruturação (REGEX) de: Raspberry.txt
Estruturação de RASPBERRY concluída. Salvo em: raspberry_estruturado.csv (Total de 74 linhas)

DataFrames ARDUINO e RASPBERRY carregados com sucesso. Iniciando Análise.

|  Média Ponderada do ARDUINO (Referência): Temp: 23.66°C | Umid: 52.42%  |

------------------------------------------------------------------------------------------
--- 🔬 Médias Ponderadas dos RASPBERRY PICO W ---
------------------------------------------------------------------------------------------
|  Média Ponderada do Pico W_11f0bb: Temp: 24.94°C | Umid: 50.65%  |
|  Média Ponderada do Pico W_11f60f: Temp: 25.08°C | Umid: 50.38%  |
-------------------------------------

/tmp/ipython-input-971618742.py:96: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  df['Timestamp'] = pd.to_datetime(df['Timestamp'], errors='coerce', infer_datetime_format=True)
